# Kredit riski üzrə ilkin məlumat təhlili

Mənbə Kaggle-ın «Give Me Some Credit» layihəsinin `data/cs-training.csv` faylıdır. Layihənin kökündə `Data Dictionary.xls`, `cs-test.csv`, `sampleEntry.csv` və `GiveMeSomeCredit.zip` də mövcuddur. `dashboard/` qovluğu ilkin baxış zamanı boş idi. Hesablamalar təlim faylı üzərində aparılıb.

Xanaları layihənin kökündən ardıcıllıqla icra edin. Aşağıdakı xülasə bu fayl üzrə aparılmış hesablamaları əks etdirir.

## 1. Məlumatın ölçüsü, sütunları və boş dəyərlər

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

data = pd.read_csv(Path("data") / "cs-training.csv")
row_count, column_count = data.shape
missing = data.isna()
schema = pd.DataFrame({
    "Məlumat tipi": data.dtypes.astype(str),
    "Boş dəyər sayı": missing.sum(),
    "Boş dəyərlərin payı (%)": (missing.mean() * 100).round(3),
})
print("Sətir sayı:", f"{row_count:,}".replace(",", " "))
print("Sütun sayı:", column_count)
print(schema.to_string())
print(f"Ən azı bir boş dəyəri olan sətir sayı: {missing.any(axis=1).sum()}")
print(f"Hər iki göstəricisi boş olan sətir sayı: {missing[['MonthlyIncome', 'NumberOfDependents']].all(axis=1).sum()}")

Sətir sayı: 150 000
Sütun sayı: 12
                                     Məlumat tipi  Boş dəyər sayı  Boş dəyərlərin payı (%)
Unnamed: 0                                  int64               0                    0.000
SeriousDlqin2yrs                            int64               0                    0.000
RevolvingUtilizationOfUnsecuredLines      float64               0                    0.000
age                                         int64               0                    0.000
NumberOfTime30-59DaysPastDueNotWorse        int64               0                    0.000
DebtRatio                                 float64               0                    0.000
MonthlyIncome                             float64           29731                   19.821
NumberOfOpenCreditLinesAndLoans             int64               0                    0.000
NumberOfTimes90DaysLate                     int64               0                    0.000
NumberRealEstateLoansOrLines                int64      

Sütunların səkkizi `int64`, dördü `float64` tipində oxunur. `NumberOfDependents` sütununda boş dəyərlər olduğuna görə Pandas onu `float64` kimi oxuyur. Boşluqların necə yarandığını bu yoxlama ilə müəyyən etmək mümkün deyil. Gəliri boş olan sətirlərin çıxarılması təlim nümunəsinin təxminən beşdə birini kənarlaşdırar və nümunənin tərkibinə təsir göstərə bilər.

## 2. Təkrarlanan qeydlər

In [2]:
identifier = "Unnamed: 0"
target = "SeriousDlqin2yrs"
analysis_columns = data.columns.drop(identifier)
feature_columns = analysis_columns.drop(target)

# Hər təkrar qrupunda ilk sətirdən sonrakı sətirlər sayılır.
print(f"Unikal sətir identifikatorlarının sayı: {data[identifier].nunique()}")
print(f"Bütün sütunlar üzrə təkrar sətir sayı: {data.duplicated().sum()}")
print(f"İdentifikator xaric təkrar sətir sayı: {data.duplicated(subset=analysis_columns).sum()}")
print(f"Yalnız izahlı dəyişənlər üzrə təkrar sətir sayı: {data.duplicated(subset=feature_columns).sum()}")

Unikal sətir identifikatorlarının sayı: 150000
Bütün sütunlar üzrə təkrar sətir sayı: 0
İdentifikator xaric təkrar sətir sayı: 609
Yalnız izahlı dəyişənlər üzrə təkrar sətir sayı: 646


CSV başlığındakı adsız ilk sütun Pandas-da `Unnamed: 0` kimi görünür və sətir identifikatoru kimi götürülüb. Eyni göstəricilərə malik sətirlər müxtəlif şəxslərə aid ola bilər. Təkrarların avtomatik silinməsi belə müşahidələri də nümunədən çıxara bilər.

## 3. Hədəf dəyişəninin paylanması

In [3]:
target_counts = data[target].value_counts(dropna=False).sort_index()
target_summary = pd.DataFrame({
    "Sətir sayı": target_counts,
    "Pay (%)": (target_counts / row_count * 100).round(3),
})
print(target_summary.to_string())
print(f"0 və 1 xaricində olan, o cümlədən boş hədəf dəyərlərinin sayı: {(~data[target].isin([0, 1])).sum()}")

                  Sətir sayı  Pay (%)
SeriousDlqin2yrs                     
0                     139974   93.316
1                      10026    6.684
0 və 1 xaricində olan, o cümlədən boş hədəf dəyərlərinin sayı: 0


`SeriousDlqin2yrs` iki il ərzində ciddi ödəniş gecikməsi ilə bağlı ikili göstəricidir. `1` hadisənin qeydə alındığını, `0` isə qeydə alınmadığını göstərir. Hadisənin payı 6,684% olduğuna görə siniflər arasında ciddi say fərqi var.

## 4. Şübhəli və kənar dəyərlər

Minimum, median, 95-ci və 99-cu persentillər və maksimum göstəricilərin miqyasını qiymətləndirməyə kömək edir. Nisbətlər üçün 1 həddi 100%-ə uyğundur və burada ilkin baxış üçün istifadə edilir.

In [4]:
features = data[feature_columns]
profile = features.describe(percentiles=[0.5, 0.95, 0.99]).T
profile = profile[["min", "50%", "95%", "99%", "max"]]
profile.columns = ["Minimum", "Median", "95-ci persentil", "99-cu persentil", "Maksimum"]
print(profile.round(4).to_string())

checks = pd.Series({
    "Yaşı 0 olan sətirlər": data["age"].eq(0).sum(),
    "Aylıq gəliri 0 olan sətirlər": data["MonthlyIncome"].eq(0).sum(),
    "Borclanma nisbəti 1-dən böyük olan sətirlər": data["DebtRatio"].gt(1).sum(),
    "Təminatsız kredit limiti üzrə istifadə nisbəti 1-dən böyük olan sətirlər":
        data["RevolvingUtilizationOfUnsecuredLines"].gt(1).sum(),
    "Mənfi dəyər olan sətirlər": features.lt(0).any(axis=1).sum(),
    "Sonsuz dəyər olan sətirlər": np.isinf(features).any(axis=1).sum(),
})
print()
print(checks.to_string())

delinquency_columns = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate",
]
delinquency = data[delinquency_columns]
for value in (96, 98):
    matches = delinquency.eq(value)
    print()
    print(f"{value} qiymətinin sütunlar üzrə sayı:")
    print(matches.sum().to_string())
    print(f"Hər üç sütunda eyni vaxtda: {matches.all(axis=1).sum()}")

                                      Minimum     Median  95-ci persentil  99-cu persentil   Maksimum
RevolvingUtilizationOfUnsecuredLines      0.0     0.1542              1.0            1.093    50708.0
age                                       0.0    52.0000             78.0           87.000      109.0
NumberOfTime30-59DaysPastDueNotWorse      0.0     0.0000              2.0            4.000       98.0
DebtRatio                                 0.0     0.3665           2449.0         4979.040   329664.0
MonthlyIncome                             0.0  5400.0000          14587.6        25000.000  3008750.0
NumberOfOpenCreditLinesAndLoans           0.0     8.0000             18.0           24.000       58.0
NumberOfTimes90DaysLate                   0.0     0.0000              1.0            3.000       98.0
NumberRealEstateLoansOrLines              0.0     1.0000              3.0            4.000       54.0
NumberOfTime60-89DaysPastDueNotWorse      0.0     0.0000              1.0         


Yaşı 0 olan sətirlər                                                            1
Aylıq gəliri 0 olan sətirlər                                                 1634
Borclanma nisbəti 1-dən böyük olan sətirlər                                 35137
Təminatsız kredit limiti üzrə istifadə nisbəti 1-dən böyük olan sətirlər     3321
Mənfi dəyər olan sətirlər                                                       0
Sonsuz dəyər olan sətirlər                                                      0

96 qiymətinin sütunlar üzrə sayı:
NumberOfTime30-59DaysPastDueNotWorse    5
NumberOfTime60-89DaysPastDueNotWorse    5
NumberOfTimes90DaysLate                 5
Hər üç sütunda eyni vaxtda: 5

98 qiymətinin sütunlar üzrə sayı:
NumberOfTime30-59DaysPastDueNotWorse    264
NumberOfTime60-89DaysPastDueNotWorse    264
NumberOfTimes90DaysLate                 264
Hər üç sütunda eyni vaxtda: 264


## Əsas analitik nəticələr

- Təlim faylında **150 000 sətir və 12 sütun** var. `MonthlyIncome` sütununda **29 731** (19,821%), `NumberOfDependents` sütununda **3 924** (2,616%) dəyər çatışmır. İkinci qrupun bütün sətirlərində aylıq gəlir də boşdur.
- Bütün sütunlar üzrə tam təkrar yoxdur. Sətir identifikatoru nəzərə alınmadıqda **609** təkrar sətir, hədəf dəyişəni də nəzərə alınmadıqda **646** təkrar sətir aşkarlanıb.
- Hədəf dəyişəni **139 974** sətirdə `0` (93,316%), **10 026** sətirdə `1` (6,684%) qiymətini alır.
- **Bir sətirdə yaş 0-dır** və bu qeyd yoxlama tələb edir. **1 634** sətirdə aylıq gəlir 0-dır. Sıfır gəlir mümkün olduğundan onun boş dəyər kimi kodlanması üçün əlavə əsas lazımdır. Ən yüksək aylıq gəlir **3 008 750**, median isə **5 400**-dür.
- Üç gecikmə sütununda **5 sətirdə 96**, **264 sətirdə 98** qiyməti eyni vaxtda görünür. Bu nümunə xüsusi kodlaşdırma ehtimalını yaradır. Məna təsdiqlənənədək həmin qiymətlər gecikmə sayı üzrə ortalamanı şişirdə bilər.
- `DebtRatio` **35 137**, təminatsız kredit limiti üzrə istifadə nisbəti **3 321** sətirdə 1-dən böyükdür. Maksimumlar müvafiq olaraq **329 664** və **50 708**-dir. Belə yüksək qiymətlər ortalamalara ciddi təsir göstərə bilər. Nisbətin 1-i keçməsi özlüyündə məlumat xətasını müəyyən etməyə kifayət etmir.

Nəticələr yalnız bu təlim nümunəsini təsvir edir və Azərbaycan bank sektoruna şamil edilə bilməz.

## 5. İşçi məlumatın təmizlənməsi

Təmizləmə və qruplaşdırma qaydaları `prepare_data.py` faylında saxlanılır. Pano və bu dəftər eyni qaydalardan istifadə edir.

| Qərar | Səbəb və mümkün təsir |
| --- | --- |
| Adsız identifikator analitik sütunlardan çıxarıldı. | Bu sütun sətirləri fərqləndirir. Müşahidə sayı və hədəf dəyişəni qorunur. |
| Yaşın aşağı həddi 18 götürüldü, bu həddən aşağı yaş naməlum kimi işarələndi. | Faylda yalnız 0 yaşlı bir sətir bu qaydaya düşür. Həmin sətir ümumi gecikmə payında qalır və orta yaşın məxrəcinə daxil edilmir. |
| Üç gecikmə sütununda 96 və 98 qiymətləri naməlum kimi işarələndi. | Eyni sətirlərdə üst-üstə düşən bu qiymətlər şübhəlidir. Mənası təsdiqlənənədək tarixçə naməlum qrupa düşür. Sətirlər və hədəf dəyərləri saxlanılır. |
| Boş gəlir və himayədə olan şəxslərin sayı doldurulmadı. Sıfır gəlir saxlanıldı. | Çatışmayan dəyərləri orta və ya sıfırla əvəz etmək paylanmanı dəyişər. Orta gəlirin məxrəci gəliri məlum müşahidələrdir. |
| Mənfi və sonsuz dəyərlər naməlum kimi işarələnir. 100%-dən yuxarı sonlu nisbətlər ayrıca qrupda saxlanılır. | Təlim faylında mənfi və sonsuz dəyər yoxdur. Yüksək nisbətlərin xəta olduğu təsdiqlənməyib. |
| Təkrar analitik profillər saxlanıldı. | Şəxs səviyyəsində eyniliyi təsdiqləyən açar yoxdur. Belə sətirlərin silinməsi nümunənin tərkibini və gecikmə payını dəyişə bilər. |

İşçi məlumat operativ yaddaşda hazırlanır və orijinal CSV üzərində yazılmır.

In [5]:
import sys

from prepare_data import DIMENSIONS, LABELS, LATE, TARGET, clean_data, segment_data, summarize

clean = clean_data(data)
groups = segment_data(clean)
grouped = summarize(clean, groups)

assert len(data) == len(clean) == grouped["count"].sum()
assert data[TARGET].equals(clean[TARGET])
assert grouped["events"].sum() == data[TARGET].sum()

audit = pd.DataFrame({
    "İlkin boş dəyər sayı": data[clean.columns].isna().sum(),
    "Təmizləmədən sonra boş dəyər sayı": clean.isna().sum(),
})
display(audit.loc[audit.ne(0).any(axis=1)])
print("İcra mühiti:", sys.executable)
print("Pandas versiyası:", pd.__version__)
print("NumPy versiyası:", np.__version__)

,İlkin boş dəyər sayı,Təmizləmədən sonra boş dəyər sayı
age,0,1
NumberOfTime30-59DaysPastDueNotWorse,0,269
MonthlyIncome,29731,29731
NumberOfTimes90DaysLate,0,269
NumberOfTime60-89DaysPastDueNotWorse,0,269
NumberOfDependents,3924,3924


İcra mühiti: C:\ProgramData\anaconda3\python.exe
Pandas versiyası: 2.3.3
NumPy versiyası: 2.3.5


## 6. Biznes təhlili

Hər qrup üzrə ciddi gecikmə payı hədəf qiyməti 1 olan müşahidələrin qrupdakı bütün müşahidələrə nisbətidir. Hər payın yanında qrupun ölçüsü göstərilir. Kiçik qruplardakı paylar az sayda hadisədən kəskin təsirlənə bilər.

Gecikmə tarixçəsi üç sütundakı sayların cəmi üzrə qruplaşdırılır. Sütunlardan hər hansı biri boşdursa, cəm naməlum sayılır. Qruplar arasındakı fərqlər təsviri xarakter daşıyır və səbəb-nəticə əlaqəsini müəyyən etmir.

In [6]:
names = {
    "age": "Yaş qrupu",
    "income": "Aylıq gəlir",
    "late": "Gecikmə tarixçəsi",
    "utilization": "Təminatsız limitdən istifadə",
    "debt": "Borclanma nisbəti",
}
for key in DIMENSIONS:
    segment = grouped.groupby(key)[["count", "events"]].sum()
    segment = segment.reindex(range(len(LABELS[key])), fill_value=0)
    segment["Ciddi gecikmə payı (%)"] = segment["events"].div(segment["count"].replace(0, np.nan)).mul(100)
    segment.index = pd.Index(LABELS[key], name=names[key])
    display(segment.rename(columns={"count": "Müşahidə sayı", "events": "Ciddi gecikmə sayı"}).round(3))

,Müşahidə sayı,Ciddi gecikmə sayı,Ciddi gecikmə payı (%)
Yaş qrupu,,,
18–29,8820,1035,11.735
30–44,38982,3700,9.492
45–59,53879,3791,7.036
60–74,36948,1271,3.440
75 və yuxarı,11370,229,2.014
Naməlum,1,0,0.000


,Müşahidə sayı,Ciddi gecikmə sayı,Ciddi gecikmə payı (%)
Aylıq gəlir,,,
0,1634,66,4.039
1–2 999,21692,2041,9.409
3 000–5 999,43814,3496,7.979
6 000–9 999,33344,1898,5.692
10 000 və yuxarı,19785,856,4.327
Naməlum,29731,1669,5.614


,Müşahidə sayı,Ciddi gecikmə sayı,Ciddi gecikmə payı (%)
Gecikmə tarixçəsi,,,
Gecikmə yoxdur,119637,3264,2.728
1 gecikmə,17243,2106,12.214
2–3 gecikmə,8826,2439,27.634
4 və daha çox,4025,2070,51.429
Naməlum,269,147,54.647


,Müşahidə sayı,Ciddi gecikmə sayı,Ciddi gecikmə payı (%)
Təminatsız limitdən istifadə,,,
25%-dən az,87657,1873,2.137
25%–50%-dən az,21055,1114,5.291
50%–100%,37967,5802,15.282
100%-dən çox,3321,1237,37.248
Naməlum,0,0,NaN


,Müşahidə sayı,Ciddi gecikmə sayı,Ciddi gecikmə payı (%)
Borclanma nisbəti,,,
25%-dən az,52357,3125,5.969
25%–50%-dən az,41345,2530,6.119
50%–100%,21161,2080,9.829
100%-dən çox,35137,2291,6.520
Naməlum,0,0,NaN


## 7. Nəticələr və həssaslıq yoxlaması

Gəlir üzrə orta ilə medianın müqayisəsi böyük qiymətlərin ortalamaya təsirini göstərməyə kömək edir. Eyni qaydada, naməlum gecikmə tarixçəsi olan müşahidələrin paya təsiri onların hesablamadan çıxarıldığı ssenari ilə müqayisə olunur. Bu ssenari panodakı əsas göstəricini dəyişmir.

In [7]:
def az_number(value, decimals=0):
    return f"{value:,.{decimals}f}".replace(",", " ").replace(".", ",")

late_history = grouped.groupby("late")[["count", "events"]].sum()
none = late_history.loc[0]
frequent = late_history.loc[3]
known_late = clean[LATE].notna().all(axis=1)
income = clean["MonthlyIncome"]

print(f"Gecikmə qeydə alınmayan qrupda ciddi gecikmə payı {az_number(none.events / none['count'] * 100, 2)}% təşkil edir. Bu qrupda {az_number(none['count'])} müşahidə var.")
print(f"Tarixçəsində 4 və daha çox gecikmə olan qrupda bu pay {az_number(frequent.events / frequent['count'] * 100, 2)}%, müşahidə sayı isə {az_number(frequent['count'])}-dir.")
print(f"Aylıq gəlirin ortası {az_number(income.mean(), 2)}, medianı {az_number(income.median())} təşkil edir.")
print(f"Ümumi ciddi gecikmə payı {az_number(clean[TARGET].mean() * 100, 3)}%, gecikmə tarixçəsi məlum olan müşahidələrdə isə {az_number(clean.loc[known_late, TARGET].mean() * 100, 3)}% təşkil edir.")
print(f"Yaş və gecikmə tarixçəsi üzrə dəyişikliklərdən sonra bütün {az_number(len(clean))} müşahidə saxlanılıb.")

Gecikmə qeydə alınmayan qrupda ciddi gecikmə payı 2,73% təşkil edir. Bu qrupda 119 637 müşahidə var.
Tarixçəsində 4 və daha çox gecikmə olan qrupda bu pay 51,43%, müşahidə sayı isə 4 025-dir.
Aylıq gəlirin ortası 6 670,22, medianı 5 400 təşkil edir.
Ümumi ciddi gecikmə payı 6,684%, gecikmə tarixçəsi məlum olan müşahidələrdə isə 6,598% təşkil edir.
Yaş və gecikmə tarixçəsi üzrə dəyişikliklərdən sonra bütün 150 000 müşahidə saxlanılıb.


Gecikmə tarixçəsi daha yüksək ciddi gecikmə payı müşahidə olunan qrupları ayırmağa kömək edir. Aylıq gəlirin orta qiyməti medianı üstələyir və gəliri məlum müşahidələrin paylanması şərh edilərkən hər iki göstərici nəzərə alınmalıdır. Hesablamalar Kaggle təlim nümunəsinə aiddir.